<a href="https://colab.research.google.com/github/arreseb/Estrutura-de-dados/blob/main/Exerc%C3%ADcio_Simulado_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Exercício de Simulação: Análise de Fluxo e Eficiência Previdenciária**

###**1. Estrutura do Notebook**
Estruturaremos o Google Collab da seguinte forma:

* **Etapa I: Entendimento do(s) dataset(s) e Perguntas de negócio a serem respondidas**

  Onde definiremos o que queremos descobrir.

* **Etapa II: Preparação e Limpeza**

  Utilizaremos o [pandas](https://pandas.pydata.org/) para tratar os dados brutos e [numpy](https://numpy.org/) para manipulação de números e arrays.

* **Etapa III: Exploração Visual (EDA)**

  Usaremos [seaborn](https://seaborn.pydata.org/) ou [plotly](https://plotly.com/python/) para identificar padrões.

* **Etapa IV: Identificação de Padrões e Correlações**

  Aqui devemos gerar, por exemplo, uma matriz de correlação para entender como alguma variável afetadeterminado parâmetro.

### ❎ **Etapa I. O Dataset (Contexto de Negócio)**

Para este primeiro exercício, não vamos trabalhar com um arquivo/planilha, vamos trabalhar criando um dataset fictício.

Imaginem que recebemos uma extração do sistema de agendamentos e concessões. O dataset contém as seguintes colunas:

* **ID_Requerimento:** Identificador único.
* **Data_Entrada:** Data do pedido.
* **Data_Conclusao**: Data da decisão final.
* **Especie_Beneficio:** Tipo (Aposentadoria, Auxílio-Doença, BPC).
* **Status:** Concedido ou Indeferido.
* **Unidade_Federativa (UF):** Estado da agência.
* **Pontuacao_Complexidade:** Score de 1 a 10 da dificuldade do processo.

Neste primeiro exercício, vamos gerar um CSV fictício com Python, conforme código abaixo:

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Criando dados sintéticos
np.random.seed(42)
n_rows = 1000

data = {
    'id_requerimento': range(1, n_rows + 1),
    'data_entrada': [datetime(2023, 1, 1) + timedelta(days=np.random.randint(0, 180)) for _ in range(n_rows)],
    'especie': np.random.choice(['Aposentadoria', 'Auxílio-Doença', 'BPC', 'Pensão'], n_rows),
    'uf': np.random.choice(['SP', 'RJ', 'MG', 'CE', 'AM', 'RS'], n_rows),
    'status': np.random.choice(['Concedido', 'Indeferido', 'Em Análise'], n_rows, p=[0.45, 0.45, 0.1]),
    'complexidade_score': np.random.randint(1, 11, n_rows),
    'idade_requerente': np.random.randint(18, 85, n_rows)
}

df = pd.DataFrame(data)

# Simulando data de conclusão (com alguns valores nulos para "Em Análise")
df['data_conclusao'] = df.apply(lambda x: x['data_entrada'] + timedelta(days=np.random.randint(15, 120))
                               if x['status'] != 'Em Análise' else pd.NaT, axis=1)

# Inserindo alguns "erros" para limpeza (Outliers e NaNs)
df.loc[0:10, 'complexidade_score'] = 99  # Erro de digitação
print("Dataset 'base_previdencia.csv' gerado com sucesso!")

Dataset 'base_previdencia.csv' gerado com sucesso!


Vamos agora salvar o conteúdo fictício criado anteriormente em um dataframe (tabela do pandas)

In [ ]:
df.to_csv('base_previdencia.csv', index=False)
print("DataFrame 'df' salvo como 'base_previdencia.csv' com sucesso!")

DataFrame 'df' salvo como 'base_previdencia.csv' com sucesso!


Vamos conhecer o nosso dataframe e seus dados:

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

### ❎ **Etapa II: Limpeza e Transformação**
Precisamos tratar os dados brutos.

* **Tratar Outliers:** O que fazer com o complexidade_score = 99? (Sugestão: substituir pela mediana ou excluir).

* **Feature Engineering**: Criem a coluna tempo_espera (data_conclusao - data_entrada).

* **Tratar Nulos:** Como lidar com os processos "Em Análise" que não possuem data de conclusão?

In [ ]:
import pandas as pd

# 1. Carregar o DataFrame
df = pd.read_csv('base_previdencia.csv')

# 2. Tratar Outliers no complexidade_score (Substituir 99 pela mediana)
mediana_score = df[df['complexidade_score'] != 99]['complexidade_score'].median()
df['complexidade_score'] = df['complexidade_score'].replace(99, mediana_score)

# 3. Converter colunas para o formato de data (datetime)
df['data_entrada'] = pd.to_datetime(df['data_entrada'])
df['data_conclusao'] = pd.to_datetime(df['data_conclusao'])

# 4. Feature Engineering: Criar coluna tempo_espera (em dias)
# O resultado para processos "Em Análise" será NaN (correto matematicamente)
df['tempo_espera'] = (df['data_conclusao'] - df['data_entrada']).dt.days

# Visualizar o resultado
print(df[['id_requerimento', 'status', 'tempo_espera']].head())

# Salvar o arquivo limpo
df.to_csv('base_previdencia_limpa.csv', index=False)

### ❎ **Etapa III: Análise Exploratória Visual (EDA)**

Para esta etapa de Análise Exploratória Visual (EDA), utilizaremos as bibliotecas Seaborn e Matplotlib para gerar os gráficos. Abaixo estão as análises baseadas nos dados transformados:

**1. Histograma do Tempo de Espera**
O histograma mostra como os tempos de concessão estão distribuídos.

* Observação: A distribuição parece ser relativamente uniforme/multimodal, com vários picos entre 20 e 110 dias. Não segue uma "curva normal" perfeita, o que indica que não há um tempo "padrão" muito forte; os processos variam bastante dentro dessa janela de 15 a 120 dias.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Configuração visual
sns.set_theme(style="whitegrid")

# 1. Histograma
plt.figure(figsize=(10, 6))
sns.histplot(df['tempo_espera'].dropna(), kde=True, color='skyblue')
plt.title('Distribuição do Tempo de Espera (Dias)')
plt.savefig('histograma_tempo_espera.png')

**2. Boxplot de Tempo vs. Espécie**

Este gráfico é excelente para identificar gargalos por tipo de benefício.

O que observar: A linha central dentro de cada caixa é a mediana. Se uma espécie (como BPC ou Aposentadoria) tiver a caixa mais "alta" no eixo Y, significa que ela tende a demorar mais.

Resultado: No gráfico gerado, as espécies possuem medianas muito próximas (em torno de 65-70 dias), sugerindo que o tipo de benefício não é o fator determinante para o atraso nesta base específica.

In [ ]:
# 2. Boxplot por Espécie
plt.figure(figsize=(12, 6))
sns.boxplot(x='especie', y='tempo_espera', data=df, palette='viridis')
plt.title('Tempo de Espera por Espécie de Benefício')
plt.xticks(rotation=45)
plt.savefig('boxplot_tempo_especie.png')

O **Boxplot (ou Diagrama de Caixa)** é uma das ferramentas mais poderosas da estatística visual porque consegue resumir, em um único desenho, cinco informações cruciais sobre como os seus dados estão espalhados.

**1. A Anatomia do Boxplot**
Imagine a "caixa" dividida em partes:

* **A Linha Central (Mediana):** É o "coração" do gráfico. Ela indica que 50% dos seus processos demoraram menos que aquele valor e 50% demoraram mais. Se a linha está no 60, a metade da fila anda em até 60 dias.

* **O Corpo da Caixa (IQR):** Representa os 50% centrais dos dados (do 25º ao 75º percentil). Se a caixa é comprida, significa que os tempos de espera variam muito. Se é achatada, os processos têm tempos muito parecidos.

* **As "Antenas" (Whiskers):** Mostram a dispersão para os valores mais altos e baixos, excluindo os pontos muito fora da curva.

* **Os Pontos Soltos (Outliers):** São aqueles processos "atípicos" que demoraram muito mais (ou muito menos) que o normal daquela categoria.

**2. Os Achados nos seus Dados (Tempo vs. Espécie)**

Ao olhar para o gráfico que compara BPC, Aposentadoria, Pensão e Auxílio-Doença, extraímos as seguintes conclusões sobre a sua base:

**Equilíbrio de Filas**

As medianas de todas as espécies estão em um nível muito similar (entre 60 e 70 dias).

* **O que isso diz:** Não existe um "privilégio" claro. O INSS não parece estar despachando Aposentadorias muito mais rápido que Pensões, por exemplo. O esforço parece ser distribuído igualmente entre as espécies.

**Variabilidade Consistente**

Todas as "caixas" possuem tamanhos parecidos.

* **O que isso diz:** A incerteza é a mesma para qualquer benefício. Seja qual for o pedido, o cidadão tem a mesma chance de ter um processo "vapt-vupt" ou um processo que se arrasta por meses.

**Ausência de Outliers Extremos**

Como limitamos os dados e tratamos os erros antes, as antenas capturam bem a variação.

* **O que isso diz:** O sistema é previsível dentro da sua própria demora. Não temos casos que levaram 2 anos enquanto outros levaram 10 dias; a maioria está contida no intervalo de 15 a 120 dias.

**3. Por que o Boxplot é melhor que a Média?**

Se eu te dissesse apenas que a "média de espera é 67 dias", eu estaria escondendo que alguns processos levam 110 dias. O Boxplot te mostra o risco: ele revela que, embora a média seja 67, a "caixa" vai até quase 100 dias.

### ❎ **Etapa IV: Conclusão e Insights Gerenciais**

Não terminem o trabalho com código. O Professor valoriza a interpretação.

Pergunta: "O que os dados dizem sobre a eficiência operacional?"

Resposta esperada: "Identificamos que a UF X possui um TMC 20% superior à média nacional. Recomendamos uma força-tarefa de análise documental nessa região."

**1. Matriz de Correlação**

A matriz de correlação mede a força da relação entre as variáveis (de -1 a 1).

Resultados Reais:

* **complexidade_score vs tempo_espera:** 0.01 (Quase zero).
* **idade_requerente vs tempo_espera:** 0.03 (Muito baixa).

Conclusão: Surpreendentemente, nesta base de dados, nem a idade do requerente nem a complexidade do caso influenciam o tempo de espera. Isso sugere que o tempo de conclusão pode estar mais ligado a fatores externos (como volume de demanda na UF ou eficiência do posto) do que às características do processo em si.

In [ ]:
# 3. Matriz de Correlação
plt.figure(figsize=(8, 6))
corr = df[['complexidade_score', 'idade_requerente', 'tempo_espera']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title('Matriz de Correlação')
plt.savefig('matriz_correlacao.png')

A **Matriz de Correlação** é uma ferramenta estatística que mede a força e a direção da relação linear entre duas variáveis numéricas. O resultado é sempre um número entre -1 e 1, chamado de Coeficiente de Correlação de Pearson.

Aqui está como interpretar esses valores:

* **1.0 (Correlação Positiva Perfeita):** Quando uma variável aumenta, a outra aumenta na mesma proporção.

* **0.0 (Nenhuma Correlação):** As variáveis não possuem relação linear entre si. O movimento de uma não diz nada sobre a outra.

* **-1.0 (Correlação Negativa Perfeita):** Quando uma variável aumenta, a outra diminui proporcionalmente.

**Os Achados na sua Base de Previdência**

Ao analisarmos a sua matriz especificamente, os números revelaram um cenário interessante e, de certa forma, contraintuitivo para o senso comum. Vamos aos detalhes:

**1. Complexidade vs. Tempo de Espera (0.0099)**

O que significa: O valor é praticamente zero.

* A conclusão: No mundo ideal, processos mais complexos deveriam demorar mais. No entanto, os seus dados mostram que a complexidade do caso não está influenciando o tempo que ele leva para ser concluído. Um caso simples (score 1) pode demorar tanto quanto um caso complexo (score 10).

**2. Idade do Requerente vs. Tempo de Espera (0.0305)**

O que significa: Outro valor extremamente baixo, quase nulo.

* A conclusão: Não há priorização ou atraso sistemático baseado na idade nesta amostra. Idosos e jovens estão esperando tempos muito semelhantes na fila.

**3. Idade vs. Complexidade (0.0011)**

O que significa: Zero absoluto.

* A conclusão: Não existe relação entre a idade da pessoa e o quão complexo é o seu processo previdenciário.

In [ ]:
import plotly.express as px
import pandas as pd

# Gráfico de Dispersão Interativo
df_limpo = pd.read_csv('/content/base_previdencia_limpa.csv')
fig1 = px.scatter(df_limpo.dropna(),
                 x="complexidade_score",
                 y="tempo_espera",
                 color="status",
                 color_discrete_map={
                     'Concedido': 'blue',
                     'Indeferido': 'red',
                     'Em Análise': 'orange'
                 },
                 hover_data=['id_requerimento', 'especie', 'uf'],
                 title="Análise Interativa: Complexidade vs. Tempo de Espera",
                 labels={"complexidade_score": "Complexidade (1-10)", "tempo_espera": "Dias de Espera"})

fig1.show()

O **gráfico de dispersão interativo** mostra a relação entre a complexidade do processo e o tempo de espera, com cores indicando o status (Concedido, Indeferido, Em Análise). Você pode passar o mouse sobre os pontos para ver detalhes de cada requerimento, como ID, espécie e UF.

Este gráfico visualmente reforça o que a matriz de correlação já indicava: não há uma relação clara entre a complexidade do processo e o tempo que ele leva para ser concluído. Os pontos estão bastante espalhados, e os processos de alta complexidade podem ter tempos de espera tanto curtos quanto longos, e o mesmo ocorre para processos de baixa complexidade. O status (Concedido, Indeferido, Em Análise) também não parece ter um padrão distinto em relação a complexidade e tempo de espera neste gráfico.

Diante do exposto recomendamos a alta gestão .......

##### ⛓ Brasília/DF, 31 de março de 2026